# 🤖 4-DOF Arm SAC Reinforcement Learning Training on Google Colab

This notebook trains a **Soft Actor-Critic (SAC)** policy for the 4-DOF manipulator in PyBullet simulation. The trained model weights are saved and downloaded as `best_sac_policy.zip` for edge deployment on the Jetson Orin Nano.

In [1]:
# Step 1: Install PyBullet, Gymnasium, Stable-Baselines3, and PyTorch dependencies
!pip install -q pybullet gymnasium stable-baselines3 torch torchvision opencv-python onnx onnxscript matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 15.6 MB/s eta 0:00:00


In [2]:
# Step 2: System & GPU Verification
import torch
import pybullet as p
import numpy as np
import os

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU (GPU acceleration recommended for long training runs)")

PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU Device: Tesla T4


In [3]:
# Step 3: Scaffold PyBullet 4-DOF Manipulator Gymnasium Environment
import gymnasium as gym
from gymnasium import spaces
import pybullet_data

# Create URDF inline for self-contained Colab execution
urdf_content = '''<?xml version="1.0"?>
<robot name="arm_4dof">
  <link name="base_link">
    <visual><geometry><cylinder length="0.1" radius="0.1"/></geometry></visual>
    <collision><geometry><cylinder length="0.1" radius="0.1"/></geometry></collision>
    <inertial><mass value="2.0"/><inertia ixx="0.01" ixy="0" ixz="0" iyy="0.01" iyz="0" izz="0.01"/></inertial>
  </link>
  <joint name="joint_1" type="revolute">
    <parent link="base_link"/><child link="link_1"/><origin xyz="0 0 0.05" rpy="0 0 0"/><axis xyz="0 0 1"/>
    <limit lower="-3.14159" upper="3.14159" effort="10.0" velocity="2.0"/>
  </joint>
  <link name="link_1">
    <visual><origin xyz="0 0 0.1"/><geometry><cylinder length="0.2" radius="0.04"/></geometry></visual>
    <collision><origin xyz="0 0 0.1"/><geometry><cylinder length="0.2" radius="0.04"/></geometry></collision>
    <inertial><mass value="0.8"/><inertia ixx="0.003" ixy="0" ixz="0" iyy="0.003" iyz="0" izz="0.001"/></inertial>
  </link>
  <joint name="joint_2" type="revolute">
    <parent link="link_1"/><child link="link_2"/><origin xyz="0 0 0.2" rpy="0 0 0"/><axis xyz="0 1 0"/>
    <limit lower="-1.5708" upper="1.5708" effort="10.0" velocity="2.0"/>
  </joint>
  <link name="link_2">
    <visual><origin xyz="0 0 0.15"/><geometry><cylinder length="0.3" radius="0.035"/></geometry></visual>
    <collision><origin xyz="0 0 0.15"/><geometry><cylinder length="0.3" radius="0.035"/></geometry></collision>
    <inertial><mass value="0.6"/><inertia ixx="0.004" ixy="0" ixz="0" iyy="0.004" iyz="0" izz="0.001"/></inertial>
  </link>
  <joint name="joint_3" type="revolute">
    <parent link="link_2"/><child link="link_3"/><origin xyz="0 0 0.3" rpy="0 0 0"/><axis xyz="0 1 0"/>
    <limit lower="-2.0" upper="2.0" effort="8.0" velocity="2.5"/>
  </joint>
  <link name="link_3">
    <visual><origin xyz="0 0 0.125"/><geometry><cylinder length="0.25" radius="0.03"/></geometry></visual>
    <collision><origin xyz="0 0 0.125"/><geometry><cylinder length="0.25" radius="0.03"/></geometry></collision>
    <inertial><mass value="0.4"/><inertia ixx="0.002" ixy="0" ixz="0" iyy="0.002" iyz="0" izz="0.0005"/></inertial>
  </link>
  <joint name="joint_4" type="revolute">
    <parent link="link_3"/><child link="ee_link"/><origin xyz="0 0 0.25" rpy="0 0 0"/><axis xyz="0 1 0"/>
    <limit lower="-1.5708" upper="1.5708" effort="5.0" velocity="3.0"/>
  </joint>
  <link name="ee_link">
    <visual><origin xyz="0 0 0.05"/><geometry><sphere radius="0.03"/></geometry></visual>
    <collision><origin xyz="0 0 0.05"/><geometry><sphere radius="0.03"/></geometry></collision>
    <inertial><mass value="0.2"/><inertia ixx="0.0005" ixy="0" ixz="0" iyy="0.0005" iyz="0" izz="0.0005"/></inertial>
  </link>
</robot>'''

os.makedirs("/tmp/sim_urdf", exist_ok=True)
urdf_path = "/tmp/sim_urdf/arm_4dof.urdf"
with open(urdf_path, "w") as f:
    f.write(urdf_content)

class ArmGraspEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.physics_client = p.connect(p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath(), physicsClientId=self.physics_client)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(4,), dtype=np.float32)
        obs_low = np.array([-np.pi] * 4 + [-10.0] * 4 + [-2.0] * 6, dtype=np.float32)
        obs_high = np.array([np.pi] * 4 + [10.0] * 4 + [2.0] * 6, dtype=np.float32)
        self.observation_space = spaces.Box(low=obs_low, high=obs_high, dtype=np.float32)
        self.joint_indices = [0, 1, 2, 3]
        self.ee_index = 3
        self.prev_action = np.zeros(4, dtype=np.float32)
        self.target_pos = np.array([0.2, 0.2, 0.3], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            np.random.seed(seed)
        self.current_step = 0
        p.resetSimulation(physicsClientId=self.physics_client)
        p.setGravity(0, 0, -9.81, physicsClientId=self.physics_client)
        p.loadURDF("plane.urdf", physicsClientId=self.physics_client)
        r = np.random.uniform(0.15, 0.45)
        theta = np.random.uniform(-np.pi / 2, np.pi / 2)
        z = np.random.uniform(0.1, 0.4)
        self.target_pos = np.array([r * np.cos(theta), r * np.sin(theta), z], dtype=np.float32)
        self.robot_id = p.loadURDF(urdf_path, basePosition=[0, 0, 0], useFixedBase=True, physicsClientId=self.physics_client)
        for joint in self.joint_indices:
            p.resetJointState(self.robot_id, joint, 0.0, 0.0, physicsClientId=self.physics_client)
        p.stepSimulation(physicsClientId=self.physics_client)
        return self._get_observation(), {}

    def step(self, action):
        self.current_step += 1
        action = np.clip(action, self.action_space.low, self.action_space.high)
        for i, joint in enumerate(self.joint_indices):
            p.setJointMotorControl2(self.robot_id, joint, p.VELOCITY_CONTROL, targetVelocity=action[i]*2.0, force=10.0, physicsClientId=self.physics_client)
        p.stepSimulation(physicsClientId=self.physics_client)
        obs = self._get_observation()
        ee_pos = obs[8:11]
        dist_to_target = np.linalg.norm(ee_pos - self.target_pos)
        reward = -dist_to_target - 0.05 * np.linalg.norm(action - self.prev_action)
        self.prev_action = action.copy()
        terminated = bool(dist_to_target < 0.03)
        truncated = bool(self.current_step >= 200)
        if terminated:
            reward += 10.0
        return obs, reward, terminated, truncated, {"distance": dist_to_target}

    def _get_observation(self):
        j_pos, j_vel = [], []
        for joint in self.joint_indices:
            st = p.getJointState(self.robot_id, joint, physicsClientId=self.physics_client)
            j_pos.append(st[0])
            j_vel.append(st[1])
        ee_state = p.getLinkState(self.robot_id, self.ee_index, physicsClientId=self.physics_client)
        ee_pos = list(ee_state[0])
        return np.concatenate([np.array(j_pos, dtype=np.float32), np.array(j_vel, dtype=np.float32), np.array(ee_pos, dtype=np.float32), self.target_pos.astype(np.float32)])

print('ArmGraspEnv successfully defined.')

ArmGraspEnv successfully defined.


In [4]:
# Step 4: Execute Soft Actor-Critic (SAC) Training Loop
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor

env = Monitor(ArmGraspEnv())
model = SAC(
    policy="MlpPolicy",
    env=env,
    learning_rate=3e-4,
    buffer_size=100000,
    batch_size=256,
    ent_coef="auto",
    gamma=0.99,
    tau=0.005,
    verbose=1,
    seed=42,
)

print("Starting SAC policy training for 100,000 timesteps...")
model.learn(total_timesteps=100000)

# Save model weights zip archive
os.makedirs("models", exist_ok=True)
save_path = "models/best_sac_policy.zip"
model.save(save_path)
print(f"SAC policy training complete! Model saved to '{save_path}'.")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cuda device
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Starting SAC policy training for 100,000 timesteps...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 200      |
|    ep_rew_mean     | -139     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 87       |
|    time_elapsed    | 9        |
|    total_timesteps | 800      |
| train/             |          |
|    actor_loss      | -8.34    |
|    critic_loss     | 0.0493   |
|    ent_coef        | 0.811    |
|    ent_coef_loss   | -1.41    |
|    learning_rate   | 0.0003   |
|    n_updates       | 699      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 200      |
|    ep_rew_mean     | -139     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 86       |
|    time_elapsed    | 18       |
|    total_timesteps | 1600     |
| train/             |          |
|    actor_loss      | -11.6    |
|    critic_loss     | 0.02     |
|    ent_coef 

In [ ]:
# Step 5: Download Trained Policy Zip File
try:
    from google.colab import files
    files.download("models/best_sac_policy.zip")
    print("Initiated download of best_sac_policy.zip to local machine.")
except Exception as e:
    print(f"Notice: Saved policy file available at 'models/best_sac_policy.zip'. ({e})")